<a href="https://colab.research.google.com/github/aounraza379/AutiSense-AI/blob/main/AutiSense_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q groq gradio openai-whisper gTTS
!apt install ffmpeg -y -q

import os, time, whisper
import gradio as gr
from gtts import gTTS
from groq import Groq

# --- SETUP ---
GROQ_API_KEY = "API_KEY_HERE"
client = Groq(api_key=GROQ_API_KEY)
stt_model = whisper.load_model("tiny")

# --- RAG KNOWLEDGE BASE ---
STRATEGY_KB = {
    "emotional": "Coaching: Validate the feeling first, then suggest a sensory break.",
    "social": "Coaching: Use 'I feel' statements and explain social cues.",
    "functional": "Coaching: Use 'First/Then' logic and offer specific choices.",
    "crisis": "Coaching: Stay calm and use very few words."
}

DAILY_TASKS = ["Brush Teeth", "Breakfast", "Lunch", "Playing Game", "Dinner", "Sleep"]
TASK_STEPS = {
    "Brush Teeth": ["1. Put paste on brush.", "2. Brush all teeth.", "3. Rinse your mouth."],
    "Breakfast": ["1. Sit at the table.", "2. Eat your food.", "3. Drink your water."],
    "Playing Game": ["1. Pick a toy.", "2. Play nicely.", "3. Clean up when done."],
    "Lunch": ["1. Wash hands.", "2. Eat your lunch.", "3. Put your plate away."],
    "Dinner": ["1. Sit with family.", "2. Try your food.", "3. Wipe your face."],
    "Sleep": ["1. Put on pajamas.", "2. Get in bed.", "3. Close your eyes."]
}

# --- SESSION WITH MULTIMODAL RAG ---
class Session:
    def __init__(self):
        self.scenario = "parent"
        self.memory = {}
        self.reset()

    def reset(self):
        self.history = []
        return [], f"Ready! Mode: {self.scenario.upper()}", None

    def update_rag_memory(self, text):
        t = text.lower()
        if "i like" in t:
            item = t.split("i like")[-1].strip().replace(".", "")
            self.memory['preference'] = item

        if any(w in t for w in ["sad", "happy", "angry", "scared"]):
            self.memory['state'] = "emotional"
        elif any(w in t for w in ["hungry", "bathroom", "brush", "eat"]):
            self.memory['state'] = "functional"
        else:
            self.memory['state'] = "social"

session = Session()

# --- TTS ---
def generate_speech(text):
    try:
        filename = f"voice_{int(time.time()*1000)}.mp3"
        tts = gTTS(text=text, lang='en')
        tts.save(filename)
        return filename
    except: return None

# --- CORE RAG LOGIC ---
def handle_interaction(text, history):
    if not text: return history, "Ready", None

    session.update_rag_memory(text)
    state = session.memory.get('state', 'social')
    strategy = STRATEGY_KB.get(state, STRATEGY_KB["social"])
    preference = session.memory.get('preference', '')

    # Specific Logic for Hungry (Proactive Choices)
    extra_instruction = ""
    if "hungry" in text.lower():
        pref_msg = f" or {preference}" if preference else ""
        extra_instruction = f" The child is hungry. Offer these choices: apple, pizza, or banana{pref_msg}. Ask what they like."

    base_rules = "- Stay in character - 1-2 short sentences - Use simple English."
    rag_context = f"Strategy: {strategy}. {extra_instruction}"

    if session.scenario == "teacher":
        role_desc = "You are Ms. Anna, a teacher. Be structured."
    elif session.scenario == "caretaker":
        role_desc = "You are a patient caretaker. Be direct."
    else:
        role_desc = "You are a loving parent. Be warm."

    system_prompt = f"{role_desc}\n{base_rules}\n{rag_context}"

    messages = [{"role": "system", "content": system_prompt}]
    for msg in history:
        messages.append({"role": msg.get("role", "user"), "content": msg.get("content", "")})
    messages.append({"role": "user", "content": text})

    try:
        res = client.chat.completions.create(model="llama-3.1-8b-instant", messages=messages)
        ai = res.choices[0].message.content
    except: ai = "I am here for you."

    audio_file = generate_speech(ai)
    new_history = history + [{"role": "user", "content": text}, {"role": "assistant", "content": ai}]
    return new_history, f"Last heard: {text}", audio_file

def process_audio(audio, history):
    if not audio: return history, "No audio", None, None
    text = stt_model.transcribe(audio)["text"].strip()
    h, f, a = handle_interaction(text, history)
    return h, f, a, None

def break_down_task(task_name, history):
    steps = TASK_STEPS.get(task_name, ["Let's do it together!"])
    ai_response = f"Let's {task_name}. " + " ".join(steps)
    audio_file = generate_speech(ai_response)

    formatted_steps = "\n".join(steps)
    new_history = history + [{"role": "user", "content": f"How do I {task_name}?"}, {"role": "assistant", "content": formatted_steps}]
    return new_history, f"Doing: {task_name}", audio_file

# --- UI ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🧠 AutiSense AI: RAG Adaptive Support")

    with gr.Row():
        mode_p = gr.Button("🏠 Parent Mode"); mode_t = gr.Button("🏫 Teacher Mode"); mode_c = gr.Button("🤝 Caretaker Mode")

    with gr.Row():
        with gr.Column(scale=3):
            chat = gr.Chatbot(type="messages", height=350)

            with gr.Row():
                btn_hungry = gr.Button("🍕 Hungry"); btn_tired = gr.Button("😴 Tired")
                btn_toilet = gr.Button("🚻 Bathroom"); btn_hurt = gr.Button("🤕 Hurt")
            with gr.Row():
                btn_sad = gr.Button("😢 Sad"); btn_happy = gr.Button("😊 Happy")
                btn_yes = gr.Button("✅ Yes"); btn_no = gr.Button("❌ No")

            mic = gr.Audio(sources=["microphone"], type="filepath", label="Record Voice")
            with gr.Row():
                submit_audio_btn = gr.Button("📤 Send & Clear", variant="primary")
                audio_playback = gr.Audio(autoplay=True, label="AI Voice Output", visible=True)

        with gr.Column(scale=2):
            feedback_label = gr.Label(value="Mode: PARENT")

            gr.Markdown("### 📅 Daily Schedule")
            for t_name in DAILY_TASKS:
                with gr.Row():
                    gr.Checkbox(label=t_name, value=False, scale=4)
                    btn_how = gr.Button("❓ How?", scale=1, size="sm")
                    btn_how.click(fn=lambda h, t=t_name: break_down_task(t, h),
                                  inputs=[chat],
                                  outputs=[chat, feedback_label, audio_playback])

            gr.Markdown("---")
            reset_btn = gr.Button("Reset Everything")

    # BINDINGS
    def btn_fn(text): return lambda h: handle_interaction(text, h)
    btns = {btn_hungry: "I am hungry", btn_tired: "I am tired", btn_toilet: "Bathroom", btn_hurt: "I am hurt",
            btn_sad: "I feel sad", btn_happy: "I feel happy", btn_yes: "Yes", btn_no: "No"}

    for btn, val in btns.items():
        btn.click(btn_fn(val), inputs=[chat], outputs=[chat, feedback_label, audio_playback])

    mode_p.click(lambda: (setattr(session, 'scenario', 'parent'), "Mode: PARENT")[1], None, feedback_label)
    mode_t.click(lambda: (setattr(session, 'scenario', 'teacher'), "Mode: TEACHER")[1], None, feedback_label)
    mode_c.click(lambda: (setattr(session, 'scenario', 'caretaker'), "Mode: CARETAKER")[1], None, feedback_label)

    submit_audio_btn.click(fn=process_audio, inputs=[mic, chat], outputs=[chat, feedback_label, audio_playback, mic])
    reset_btn.click(session.reset, None, [chat, feedback_label, audio_playback])

demo.launch()

In [ ]:
# import os
# print("Memory file exists:", os.path.exists("autisense_memory.pkl"))

In [ ]:
# import os
# os.remove("autisense_memory.pkl")
# print("Memory cleared — fresh start")